In [2]:
import torch
import torch.nn as nn
import os, copy, time
import numpy as np
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import shufflenet_v2_x1_0
from sklearn.model_selection import train_test_split

#  Dataset & DataLoaders (same as before)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []

for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 64
num_workers = min(8, os.cpu_count())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)


# Load FP32 pretrained ShuffleNet

state_dict = torch.load("/home/h5/sapi279d/shufflenetv2_weights.pth", map_location="cpu")
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

num_classes = 200
model_fp32 = shufflenet_v2_x1_0(weights=None)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, num_classes)
model_fp32.load_state_dict(new_state_dict, strict=True)
model_fp32.eval()
print("FP32 model loaded")


# AdaRound helper

def adaround_weight_tensor(weight, n_bits=8, num_iters=500, lr=1e-2):
    w = weight.detach().clone()          # ensure detached from graph
    scale = (w.max() - w.min()) / (2 ** n_bits - 1)
    zero_point = torch.round(-w.min() / scale)

    alpha = torch.zeros_like(w, requires_grad=True)
    optimizer = torch.optim.Adam([alpha], lr=lr)

    for i in range(num_iters):
        optimizer.zero_grad()
        soft_offset = torch.sigmoid(alpha)
        w_q = torch.floor(w / scale) + soft_offset
        soft_weight = w_q * scale

        # reconstruction loss
        loss = torch.nn.functional.mse_loss(soft_weight, w)
        loss.backward()
        optimizer.step()

        # detach alpha to cut graph for next iteration
        alpha = alpha.detach().requires_grad_()

        if i % (num_iters // 5) == 0 or i == num_iters - 1:
            print(f"Iter {i}: loss={loss.item():.6f}")

    with torch.no_grad():
        hard_offset = (torch.sigmoid(alpha) > 0.5).float()
        w_q = torch.floor(w / scale) + hard_offset
        quantized_weight = w_q * scale

    return quantized_weight



# Apply AdaRound

model_adaround = apply_adaround(model_fp32, n_bits=8, num_iters=500, lr=1e-2)


# Evaluation

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_ada  = evaluate_model(model_adaround, test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"AdaRound INT8 Accuracy: {acc_ada:.2f}%")


# Size + Runtime

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=20):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    return (total_time / num_batches) * 1000, (total_time / total_images) * 1000

fp32_size = get_model_size(model_fp32)
adaround_size = get_model_size(model_adaround)
print(f"FP32 size: {fp32_size:.2f} MB | AdaRound size: {adaround_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_ada, img_ms_ada   = benchmark(model_adaround, test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"AdaRound Inference: {batch_ms_ada:.2f} ms/batch | {img_ms_ada:.2f} ms/image")


✅ FP32 model loaded
🔧 AdaRound on layer: conv1.0
Iter 0: loss=0.000007
Iter 100: loss=0.000007
Iter 200: loss=0.000007
Iter 300: loss=0.000007
Iter 400: loss=0.000007
Iter 499: loss=0.000007
🔧 AdaRound on layer: stage2.0.branch1.0
Iter 0: loss=0.000013
Iter 100: loss=0.000013
Iter 200: loss=0.000013
Iter 300: loss=0.000013
Iter 400: loss=0.000013
Iter 499: loss=0.000013
🔧 AdaRound on layer: stage2.0.branch1.2
Iter 0: loss=0.000008
Iter 100: loss=0.000008
Iter 200: loss=0.000008
Iter 300: loss=0.000008
Iter 400: loss=0.000008
Iter 499: loss=0.000008
🔧 AdaRound on layer: stage2.0.branch2.0
Iter 0: loss=0.000014
Iter 100: loss=0.000014
Iter 200: loss=0.000014
Iter 300: loss=0.000014
Iter 400: loss=0.000014
Iter 499: loss=0.000014
🔧 AdaRound on layer: stage2.0.branch2.3
Iter 0: loss=0.000006
Iter 100: loss=0.000006
Iter 200: loss=0.000006
Iter 300: loss=0.000006
Iter 400: loss=0.000006
Iter 499: loss=0.000006
🔧 AdaRound on layer: stage2.0.branch2.5
Iter 0: loss=0.000016
Iter 100: loss=0.00